# Cycle types in double cosets of Young subgroups via Eulerian transition systems — computational companion

Dean Kulik, QuHarmonics Research Group (ORCID 0009-0003-3128-8828)

This notebook is self-contained. It uses only the Python standard library, reads no files, and reproduces every computed number in the paper. Cells run top to bottom.

With `RUN_LARGE = False` all checks run for $n \le 7$ in under a minute. With `RUN_LARGE = True` the enumeration checks extend to $n = 8$ and the table-only counts to $n = 9$, which takes about four minutes in total.

**Conventions.** Points are $0,\dots,n-1$ and a permutation is a tuple `p` with `p[x]` the image of `x`. A block type $\mu$ lists block sizes in decreasing order, and its blocks are consecutive runs of points. The table of `p` is `T[i][j]` $= \#\{x \in B_i : p(x) \in B_j\}$.

In [1]:
RUN_LARGE = False

import itertools, math, time
from collections import Counter, defaultdict
from functools import lru_cache
from math import factorial, comb, prod

N_ENUM  = 8 if RUN_LARGE else 7     # largest n for checks that enumerate permutations
N_TABLE = 9 if RUN_LARGE else 7     # largest n for counts computed from tables alone
print("enumeration checks up to n =", N_ENUM, "; table-only counts up to n =", N_TABLE)

enumeration checks up to n = 7 ; table-only counts up to n = 7


## 1. Core definitions

In [2]:
def partitions(n, maxpart=None):
    """Integer partitions of n with parts in decreasing order."""
    if maxpart is None:
        maxpart = n
    if n == 0:
        yield ()
        return
    for k in range(min(n, maxpart), 0, -1):
        for rest in partitions(n - k, k):
            yield (k,) + rest

def cycle_type(p):
    n = len(p); seen = [False] * n; lengths = []
    for i in range(n):
        if not seen[i]:
            L, j = 0, i
            while not seen[j]:
                seen[j] = True; j = p[j]; L += 1
            lengths.append(L)
    return tuple(sorted(lengths, reverse=True))

def inverse(p):
    q = [0] * len(p)
    for i, v in enumerate(p):
        q[v] = i
    return tuple(q)

def block_index(mu):
    b = []
    for i, d in enumerate(mu):
        b += [i] * d
    return b

def table_of(p, b, m):
    T = [[0] * m for _ in range(m)]
    for x, y in enumerate(p):
        T[b[x]][b[y]] += 1
    return tuple(map(tuple, T))

def tables(mu):
    """All square nonnegative integer matrices with row sums and column sums equal to mu."""
    m = len(mu); out = []
    def fill_row(j, left, colrem, row):
        if j == m - 1:
            if left <= colrem[j]:
                yield row + (left,)
            return
        for v in range(min(left, colrem[j]), -1, -1):
            yield from fill_row(j + 1, left - v, colrem, row + (v,))
    def rec(i, colrem, rows):
        if i == m:
            if all(c == 0 for c in colrem):
                out.append(tuple(rows))
            return
        for r in fill_row(0, mu[i], colrem, ()):
            rec(i + 1, [c - v for c, v in zip(colrem, r)], rows + [r])
    rec(0, list(mu), [])
    return out

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.parent[a] = b
    def count(self):
        return len({self.find(i) for i in range(len(self.parent))})

def fisher_yates(T):
    """|K sigma K| for the double coset with table T."""
    d = [sum(r) for r in T]
    return prod(factorial(x) for x in d) ** 2 // prod(factorial(v) for r in T for v in r)

def n_set_partitions_of_type(mu):
    n = sum(mu); c = Counter(mu)
    return factorial(n) // (prod(factorial(d) for d in mu) * prod(factorial(k) for k in c.values()))

def relabel(T, p):
    m = len(T)
    return tuple(tuple(T[p[i]][p[j]] for j in range(m)) for i in range(m))

def transpose(T):
    return tuple(zip(*T))

## 2. Eulerian circuits (BEST theorem) and transition-system counts

In [3]:
def det_int(M):
    """Exact integer determinant (Bareiss elimination)."""
    M = [list(r) for r in M]; n = len(M)
    if n == 0:
        return 1
    sign, prev = 1, 1
    for k in range(n - 1):
        if M[k][k] == 0:
            for r in range(k + 1, n):
                if M[r][k] != 0:
                    M[k], M[r] = M[r], M[k]; sign = -sign
                    break
            else:
                return 0
        for i in range(k + 1, n):
            for j in range(k + 1, n):
                M[i][j] = (M[i][j] * M[k][k] - M[i][k] * M[k][j]) // prev
        prev = M[k][k]
    return sign * M[n - 1][n - 1]

def balanced(S):
    m = len(S)
    return all(sum(S[i]) == sum(S[j][i] for j in range(m)) for i in range(m))

def connected(S):
    m = len(S)
    verts = [v for v in range(m) if sum(S[v]) > 0]
    if not verts:
        return True
    adj = {v: set() for v in verts}
    for i in range(m):
        for j in range(m):
            if S[i][j]:
                adj[i].add(j); adj[j].add(i)
    seen, stack = {verts[0]}, [verts[0]]
    while stack:
        x = stack.pop()
        for y in adj[x]:
            if y not in seen:
                seen.add(y); stack.append(y)
    return len(seen) == len(verts)

def eulerian_circuits(S):
    """ec(G_S): Eulerian circuits of the balanced multidigraph with S[i][j] distinguishable arcs i->j,
    counted as cyclic arc sequences (BEST theorem). Returns 0 if the arcs do not form one connected piece."""
    m = len(S)
    verts = [v for v in range(m) if sum(S[v]) > 0]
    if not verts:
        return 1
    if not connected(S):
        return 0
    L = [[(sum(S[i]) - S[i][i]) if i == j else -S[i][j] for j in verts] for i in verts]
    arborescences = det_int([row[1:] for row in L[1:]])
    return arborescences * prod(factorial(sum(S[v]) - 1) for v in verts)

def subtables(T, forced=None):
    m = len(T); flat = [v for r in T for v in r]
    ranges = [range(1 if k == forced else 0, v + 1) for k, v in enumerate(flat)]
    for S in itertools.product(*ranges):
        yield tuple(tuple(S[i * m:(i + 1) * m]) for i in range(m))

@lru_cache(maxsize=None)
def TS(T):
    """Theorem 5.1: dict lambda -> number of transition systems of G_T whose closed trails have lengths lambda."""
    m = len(T); flat = [v for r in T for v in r]
    if sum(flat) == 0:
        return {(): 1}
    e = next(k for k, v in enumerate(flat) if v)
    out = Counter()
    for S in subtables(T, forced=e):
        if not balanced(S):
            continue
        c = eulerian_circuits(S)
        if c == 0:
            continue
        Sflat = [v for r in S for v in r]
        w = prod(comb(t - 1, s - 1) if k == e else comb(t, s) for k, (t, s) in enumerate(zip(flat, Sflat)))
        R = tuple(tuple(T[i][j] - S[i][j] for j in range(m)) for i in range(m))
        L = sum(Sflat)
        for lam, cnt in TS(R).items():
            out[tuple(sorted(lam + (L,), reverse=True))] += w * c * cnt
    return dict(out)

def M_counts(T):
    """Theorem 3.1: dict lambda -> |M(T, lambda)| = TS_lambda(T) * prod d_i! / prod T_ij!."""
    num = prod(factorial(sum(r)) for r in T); den = prod(factorial(v) for r in T for v in r)
    out = {}
    for lam, c in TS(T).items():
        assert (c * num) % den == 0
        out[lam] = c * num // den
    return out

@lru_cache(maxsize=None)
def trail_spectra(T):
    """Corollary 5.3: cycle types realizable in the double coset, computed as closed-trail decompositions (no counting)."""
    m = len(T); flat = [v for r in T for v in r]
    if sum(flat) == 0:
        return frozenset([()])
    e = next(k for k, v in enumerate(flat) if v)
    res = set()
    for S in subtables(T, forced=e):
        if not balanced(S) or not connected(S):
            continue
        R = tuple(tuple(T[i][j] - S[i][j] for j in range(m)) for i in range(m))
        L = sum(v for r in S for v in r)
        for lam in trail_spectra(R):
            res.add(tuple(sorted(lam + (L,), reverse=True)))
    return frozenset(res)

# sanity check of the determinant routine against a direct Laplace expansion
def det_laplace(M):
    if len(M) == 1: return M[0][0]
    return sum((-1) ** j * M[0][j] * det_laplace([r[:j] + r[j + 1:] for r in M[1:]]) for j in range(len(M)))
import random
random.seed(1)
for _ in range(300):
    k = random.randint(1, 5)
    A = [[random.randint(-6, 6) for _ in range(k)] for _ in range(k)]
    assert det_int(A) == det_laplace(A)
print("determinant routine checked on 300 random integer matrices")

determinant routine checked on 300 random integer matrices


## 3. Lemma 2.1: double cosets of $K=S_\mu$ are exactly the classes of equal table, with Fisher–Yates sizes
The orbits of $K\times K$ acting by $\sigma\mapsto k\sigma k'$ are computed by union-find over all permutations, using generators of $K$ on both sides.

In [4]:
def check_double_cosets(n):
    perms = list(itertools.permutations(range(n))); idx = {p: i for i, p in enumerate(perms)}
    total_cosets = mismatches = size_errors = 0
    for mu in partitions(n):
        m = len(mu); b = block_index(mu); start = 0; gens = []
        for d in mu:
            for a in range(start, start + d - 1):
                g = list(range(n)); g[a], g[a + 1] = a + 1, a; gens.append(g)
            start += d
        uf = UnionFind(len(perms))
        for i, p in enumerate(perms):
            for g in gens:
                uf.union(i, idx[tuple(g[p[x]] for x in range(n))])
                uf.union(i, idx[tuple(p[g[x]] for x in range(n))])
        by_orbit = defaultdict(set); by_table = defaultdict(set); orbit_size = Counter()
        for i, p in enumerate(perms):
            T = table_of(p, b, m); r = uf.find(i)
            by_orbit[r].add(T); by_table[T].add(r); orbit_size[r] += 1
        mismatches += sum(len(v) > 1 for v in by_orbit.values()) + sum(len(v) > 1 for v in by_table.values())
        size_errors += sum(orbit_size[r] != fisher_yates(next(iter(Ts))) for r, Ts in by_orbit.items())
        total_cosets += len(by_orbit)
    return total_cosets, mismatches, size_errors

for n in range(1, N_ENUM + 1):
    t0 = time.time(); c, mm, se = check_double_cosets(n)
    print(f"n={n}: double cosets {c:>6}, partition mismatches {mm}, size mismatches {se}   [{time.time() - t0:.1f}s]")

n=1: double cosets      1, partition mismatches 0, size mismatches 0   [0.0s]
n=2: double cosets      3, partition mismatches 0, size mismatches 0   [0.0s]
n=3: double cosets      9, partition mismatches 0, size mismatches 0   [0.0s]
n=4: double cosets     37, partition mismatches 0, size mismatches 0   [0.0s]
n=5: double cosets    177, partition mismatches 0, size mismatches 0   [0.0s]
n=6: double cosets   1054, partition mismatches 0, size mismatches 0   [0.1s]
n=7: double cosets   7237, partition mismatches 0, size mismatches 0   [0.6s]


## 4. Theorems 3.1, 4.1 and 5.1, Corollary 5.3, Proposition 5.4: comparison with direct enumeration on every table
For every block type and every permutation, the counts $|M(T,\lambda)|$ obtained by enumeration are compared with the transition-system recursion. The single-cycle count is compared with the BEST closed form, the enumerated support with the closed-trail spectra, and the minimum number of cycles with the number of connected components.

In [5]:
def components(T):
    m = len(T); parent = list(range(m))
    def f(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i in range(m):
        for j in range(m):
            if T[i][j]:
                parent[f(i)] = f(j)
    return len({f(i) for i in range(m)})

def single_cycle_formula(T):
    d = [sum(r) for r in T]
    return eulerian_circuits(T) * prod(factorial(x) for x in d) // prod(factorial(v) for r in T for v in r)

for n in range(2, N_ENUM + 1):
    t0 = time.time()
    perms = list(itertools.permutations(range(n))); lam_of = [cycle_type(p) for p in perms]
    n_tables = bad_rec = bad_best = bad_supp = bad_min = 0
    for mu in partitions(n):
        m = len(mu); b = block_index(mu); counts = defaultdict(Counter)
        for p, lam in zip(perms, lam_of):
            counts[table_of(p, b, m)][lam] += 1
        for T, C in counts.items():
            n_tables += 1
            bad_rec += (dict(C) != M_counts(T))
            bad_best += (C.get((n,), 0) != single_cycle_formula(T))
            bad_supp += (set(C) != set(trail_spectra(T)))
            bad_min += (min(len(l) for l in C) != components(T))
    print(f"n={n}: tables {n_tables:>6} | recursion mismatches {bad_rec} | single-cycle formula mismatches {bad_best} | "
          f"support mismatches {bad_supp} | min-cycles mismatches {bad_min}   [{time.time() - t0:.1f}s]")

n=2: tables      3 | recursion mismatches 0 | single-cycle formula mismatches 0 | support mismatches 0 | min-cycles mismatches 0   [0.0s]
n=3: tables      9 | recursion mismatches 0 | single-cycle formula mismatches 0 | support mismatches 0 | min-cycles mismatches 0   [0.0s]
n=4: tables     37 | recursion mismatches 0 | single-cycle formula mismatches 0 | support mismatches 0 | min-cycles mismatches 0   [0.0s]
n=5: tables    177 | recursion mismatches 0 | single-cycle formula mismatches 0 | support mismatches 0 | min-cycles mismatches 0   [0.0s]
n=6: tables   1054 | recursion mismatches 0 | single-cycle formula mismatches 0 | support mismatches 0 | min-cycles mismatches 0   [0.3s]
n=7: tables   7237 | recursion mismatches 0 | single-cycle formula mismatches 0 | support mismatches 0 | min-cycles mismatches 0   [4.1s]


## 5. Example 3.4

In [6]:
T = ((1, 1), (1, 1))
print("TS(T) =", TS(T))
print("|M(T, lambda)| =", M_counts(T), "  sum =", sum(M_counts(T).values()), "  Fisher-Yates size =", fisher_yates(T))

TS(T) = {(2, 1, 1): 1, (3, 1): 2, (4,): 1}
|M(T, lambda)| = {(2, 1, 1): 4, (3, 1): 8, (4,): 4}   sum = 16   Fisher-Yates size = 16


## 6. Theorem 4.3 (dominant cycle), Corollaries 4.5 and 4.6, Theorem 4.7 (involutions), Theorem 5.2 (length-rooted recursion)
Each formula is compared with `TS`, which section 4 has already matched against enumeration on every table for these $n$.

In [7]:
def balanced_subtables_of_total(T, total):
    m = len(T); flat = [v for r in T for v in r]
    for S in itertools.product(*[range(0, v + 1) for v in flat]):
        if sum(S) != total:
            continue
        Sm = tuple(tuple(S[i * m:(i + 1) * m]) for i in range(m))
        if balanced(Sm):
            yield Sm, S

def minus(T, S):
    m = len(T)
    return tuple(tuple(T[i][j] - S[i][j] for j in range(m)) for i in range(m))

def dominant_cycle_formula(T, lam):
    """lam = (L) + nu with L > nu_1."""
    nu = lam[1:]; flat = [v for r in T for v in r]; total = 0
    for R, Rflat in balanced_subtables_of_total(T, sum(nu)):
        t_nu = TS(R).get(nu, 0) if nu else 1
        if t_nu:
            total += prod(comb(t, r) for t, r in zip(flat, Rflat)) * t_nu * eulerian_circuits(minus(T, R))
    return total

def hook_formula(T, k):
    m = len(T); total = 0
    for a in itertools.product(*[range(T[v][v] + 1) for v in range(m)]):
        if sum(a) == k:
            R = tuple(tuple(T[i][j] - (a[i] if i == j else 0) for j in range(m)) for i in range(m))
            total += prod(comb(T[v][v], a[v]) for v in range(m)) * eulerian_circuits(R)
    return total

def two_one_formula(T, j, k):
    """lambda = (L, 2^j, 1^k), L >= 3."""
    m = len(T); pairs = [(i, l) for i in range(m) for l in range(i + 1, m)]; total = 0
    for a in itertools.product(*[range(T[v][v] + 1) for v in range(m)]):
        if sum(a) != k:
            continue
        for c in itertools.product(*[range((T[v][v] - a[v]) // 2 + 1) for v in range(m)]):
            if sum(c) > j:
                continue
            for bb in itertools.product(*[range(min(T[i][l], T[l][i]) + 1) for i, l in pairs]):
                if sum(c) + sum(bb) != j:
                    continue
                R = [list(r) for r in T]; w = 1
                for v in range(m):
                    w *= comb(T[v][v], a[v]) * comb(T[v][v] - a[v], 2 * c[v]) * factorial(2 * c[v]) // (2 ** c[v] * factorial(c[v]))
                    R[v][v] -= a[v] + 2 * c[v]
                for (i, l), x in zip(pairs, bb):
                    w *= comb(T[i][l], x) * comb(T[l][i], x) * factorial(x); R[i][l] -= x; R[l][i] -= x
                if w:
                    total += w * eulerian_circuits(R)
    return total

def involution_formula(T, k):
    m = len(T)
    if T != transpose(T):
        return 0
    poly = {0: 1}
    for v in range(m):
        new = Counter()
        for a in range(T[v][v] + 1):
            r = T[v][v] - a
            if r % 2:
                continue
            w = comb(T[v][v], a) * prod(range(r - 1, 0, -2))   # (r-1)!! with (-1)!! = 1
            for e, c in poly.items():
                new[e + a] += c * w
        poly = new
    return prod(factorial(T[i][l]) for i in range(m) for l in range(i + 1, m)) * poly.get(k, 0)

def length_rooted_formula(T, lam):
    L = lam[0]; rest = lam[1:]; mult = lam.count(L); flat = [v for r in T for v in r]; total = 0
    for S, Sflat in balanced_subtables_of_total(T, L):
        c = eulerian_circuits(S)
        if c == 0:
            continue
        R = minus(T, S)
        t_rest = TS(R).get(rest, 0) if rest else (1 if sum(v for r in R for v in r) == 0 else 0)
        total += prod(comb(t, s) for t, s in zip(flat, Sflat)) * c * t_rest
    assert total % mult == 0
    return total // mult

t0 = time.time()
stats = Counter()
for n in range(2, 8):
    for mu in partitions(n):
        for T in tables(mu):
            ts = TS(T)
            for lam in ts:
                if len(lam) == 1 or lam[0] > lam[1]:
                    stats["dominant cases"] += 1; stats["dominant mismatches"] += dominant_cycle_formula(T, lam) != ts[lam]
                stats["length-rooted cases"] += 1; stats["length-rooted mismatches"] += length_rooted_formula(T, lam) != ts[lam]
            for k in range(0, n - 1):
                stats["hook cases"] += 1; stats["hook mismatches"] += hook_formula(T, k) != ts.get((n - k,) + (1,) * k, 0)
            for j in range(0, (n - 3) // 2 + 1):
                for k in range(0, n - 2 * j - 2):
                    lam = (n - 2 * j - k,) + (2,) * j + (1,) * k
                    stats["(L,2^j,1^k) cases"] += 1; stats["(L,2^j,1^k) mismatches"] += two_one_formula(T, j, k) != ts.get(lam, 0)
            for k in range(n % 2, n + 1, 2):
                lam = (2,) * ((n - k) // 2) + (1,) * k
                stats["involution cases"] += 1; stats["involution mismatches"] += involution_formula(T, k) != ts.get(lam, 0)
for key in ["dominant", "hook", "(L,2^j,1^k)", "involution", "length-rooted"]:
    print(f"{key:>14}: {stats[key + ' cases']:>6} cases, {stats[key + ' mismatches']} mismatches  (all tables, 2 <= n <= 7)")
print(f"[{time.time() - t0:.1f}s]")

      dominant:  12038 cases, 0 mismatches  (all tables, 2 <= n <= 7)
          hook:  49532 cases, 0 mismatches  (all tables, 2 <= n <= 7)
   (L,2^j,1^k):  72248 cases, 0 mismatches  (all tables, 2 <= n <= 7)
    involution:  33830 cases, 0 mismatches  (all tables, 2 <= n <= 7)
 length-rooted:  13706 cases, 0 mismatches  (all tables, 2 <= n <= 7)
[4.5s]


## 7. Theorem 6.2: census over all pairs $(P,\sigma)$ with $K \neq 1$ (Table 1)
Pairs are weighted by the number of set partitions of each block type.

In [8]:
for n in (5, 6, 7):
    t0 = time.time()
    perms = list(itertools.permutations(range(n))); lam_of = [cycle_type(p) for p in perms]; class_size = Counter(lam_of)
    pairs = strict = ties = ties_coset_smaller = 0; log_gain = 0.0
    for mu in partitions(n):
        if all(d == 1 for d in mu):
            continue
        weight = n_set_partitions_of_type(mu); m = len(mu); b = block_index(mu)
        tabs = [table_of(p, b, m) for p in perms]; counts = defaultdict(Counter)
        for T, lam in zip(tabs, lam_of):
            counts[T][lam] += 1
        for T, lam in zip(tabs, lam_of):
            M = counts[T][lam]; D = fisher_yates(T); Cl = class_size[lam]; smaller = min(D, Cl)
            pairs += weight
            if M < smaller:
                strict += weight
            else:
                ties += weight; ties_coset_smaller += weight if D <= Cl else 0
            log_gain += weight * (math.log2(smaller) - math.log2(M))
    print(f"n={n}: pairs {pairs:>9,} | strictly smaller {strict:>9,} | equal to minimum {ties:>6,} | "
          f"of which double coset is the smaller set {ties_coset_smaller} | mean log2(min/|M|) {log_gain / pairs:.3f}   [{time.time() - t0:.1f}s]")

n=5: pairs     6,120 | strictly smaller     5,730 | equal to minimum    390 | of which double coset is the smaller set 0 | mean log2(min/|M|) 1.193   [0.0s]
n=6: pairs   145,440 | strictly smaller   142,929 | equal to minimum  2,511 | of which double coset is the smaller set 0 | mean log2(min/|M|) 1.685   [0.0s]
n=7: pairs 4,415,040 | strictly smaller 4,396,147 | equal to minimum 18,893 | of which double coset is the smaller set 0 | mean log2(min/|M|) 2.221   [0.3s]


## 8. Section 7: orbit counts, profile counts and Theorem 7.1 (Table 2)

In [9]:
def set_partitions(n):
    def rec(i, blocks):
        if i == n:
            yield [tuple(bl) for bl in blocks]; return
        for bl in blocks:
            bl.append(i); yield from rec(i + 1, blocks); bl.pop()
        blocks.append([i]); yield from rec(i + 1, blocks); blocks.pop()
    yield from rec(0, [])

def burnside_orbits(n):
    """|X/S_n| and |X/(S_n x C_2)| for X = {(P, sigma)}, by Burnside's lemma."""
    perms = list(itertools.permutations(range(n)))
    parts = [frozenset(frozenset(bl) for bl in P) for P in set_partitions(n)]
    class_size = Counter(cycle_type(p) for p in perms); reps = {}
    for p in perms:
        reps.setdefault(cycle_type(p), p)
    s1 = s2 = 0
    for lam, g in reps.items():
        fixed_partitions = sum(1 for P in parts if frozenset(frozenset(g[x] for x in bl) for bl in P) == P)
        commuting = twisted = 0
        for s in perms:
            if all(g[s[x]] == s[g[x]] for x in range(n)):
                commuting += 1
            if all(s[g[s[x]]] == g[x] for x in range(n)):      # g s^-1 g^-1 = s
                twisted += 1
        s1 += class_size[lam] * fixed_partitions * commuting
        s2 += class_size[lam] * fixed_partitions * (commuting + twisted)
    assert s1 % factorial(n) == 0 and s2 % (2 * factorial(n)) == 0
    return s1 // factorial(n), s2 // (2 * factorial(n))

def table_classes(mu, with_transpose):
    tabs = tables(mu); index = {t: i for i, t in enumerate(tabs)}; uf = UnionFind(len(tabs)); m = len(mu)
    for t, i in index.items():
        for s in range(m - 1):
            if mu[s] == mu[s + 1]:
                p = list(range(m)); p[s], p[s + 1] = s + 1, s
                uf.union(i, index[relabel(t, p)])
        if with_transpose:
            uf.union(i, index[transpose(t)])
    reps = {}
    for t, i in index.items():
        reps.setdefault(uf.find(i), t)
    return len(tabs), list(reps.values())

def profile_counts(n):
    """labeled tables, table classes (without/with transpose), profiles (without/with transpose), distinct |M| values"""
    n_tab = c1 = c2 = p1 = p2 = 0; values = set()
    for mu in partitions(n):
        nt, reps1 = table_classes(mu, False); _, reps2 = table_classes(mu, True)
        n_tab += nt; c1 += len(reps1); c2 += len(reps2)
        p1 += sum(len(trail_spectra(t)) for t in reps1); p2 += sum(len(trail_spectra(t)) for t in reps2)
        for t in reps2:
            values |= set(M_counts(t).values())
    return n_tab, c1, c2, p1, p2, len(values)

def direct_orbits(n):
    """Direct union-find over orbit representatives: conjugation orbits, conjugation+inversion orbits,
    classes of the relation generated by conjugation and tau in M(P, sigma) (without/with inversion),
    and the number of conjugation+inversion orbits on which the profile without transpose is not constant."""
    perms = list(itertools.permutations(range(n))); idx = {p: i for i, p in enumerate(perms)}
    inv = [idx[inverse(p)] for p in perms]; lam_of = [cycle_type(p) for p in perms]
    o1 = o2 = r1 = r2 = split = 0
    for mu in partitions(n):
        m = len(mu); b = block_index(mu); blocks = []; start = 0
        for d in mu:
            blocks.append(list(range(start, start + d))); start += d
        gens = []
        for bl in blocks:
            for x, y in zip(bl, bl[1:]):
                g = list(range(n)); g[x], g[y] = y, x; gens.append(g)
        for s in range(m - 1):
            if mu[s] == mu[s + 1]:
                g = list(range(n))
                for x, y in zip(blocks[s], blocks[s + 1]):
                    g[x], g[y] = y, x
                gens.append(g)
        ufs = [UnionFind(len(perms)) for _ in range(4)]   # conj, conj+inv, conj+rel, conj+rel+inv
        cell = defaultdict(list); tab = []
        for i, p in enumerate(perms):
            T = table_of(p, b, m); tab.append(T); cell[(T, lam_of[i])].append(i)
            for g in gens:                                  # g is an involution, so g p g^-1 = g p g
                j = idx[tuple(g[p[g[x]]] for x in range(n))]
                for uf in ufs:
                    uf.union(i, j)
            ufs[1].union(i, inv[i]); ufs[3].union(i, inv[i])
        for members in cell.values():
            for j in members[1:]:
                ufs[2].union(members[0], j); ufs[3].union(members[0], j)
        o1 += ufs[0].count(); o2 += ufs[1].count(); r1 += ufs[2].count(); r2 += ufs[3].count()
        # class of each labeled table under size-preserving relabeling (union-find on tables)
        tabs = tables(mu); tindex = {t: k for k, t in enumerate(tabs)}; tuf = UnionFind(len(tabs))
        for t, k in tindex.items():
            for s in range(m - 1):
                if mu[s] == mu[s + 1]:
                    p = list(range(m)); p[s], p[s + 1] = s + 1, s
                    tuf.union(k, tindex[relabel(t, p)])
        profiles_in_orbit = defaultdict(set)
        for i, T in enumerate(tab):
            profiles_in_orbit[ufs[1].find(i)].add((tuf.find(tindex[T]), lam_of[i]))
        split += sum(len(v) > 1 for v in profiles_in_orbit.values())
    return o1, o2, r1, r2, split

print(" n | tables | classes ~ / ~t | |X/S_n| | |X/(S_n x C2)| | profiles | profiles, transpose-merged | values of F")
for n in range(1, N_TABLE + 1):
    t0 = time.time()
    b1, b2 = burnside_orbits(n)
    nt, c1, c2, p1, p2, nv = profile_counts(n)
    line = f"{n:>2} | {nt:>7,} | {c1:>5} / {c2:<5} | {b1:>7,} | {b2:>7,} | {p1:>7,} | {p2:>7,} | {nv:>4}"
    if n <= 7:
        o1, o2, r1, r2, split = direct_orbits(n)
        agree = (o1, o2, r1, r2) == (b1, b2, p1, p2)
        line += f"   direct union-find agrees: {agree}; inversion-orbits split by untransposed profile: {split}"
    print(line + f"   [{time.time() - t0:.1f}s]")

 n | tables | classes ~ / ~t | |X/S_n| | |X/(S_n x C2)| | profiles | profiles, transpose-merged | values of F
 1 |       1 |     1 / 1     |       1 |       1 |       1 |       1 |    1   direct union-find agrees: True; inversion-orbits split by untransposed profile: 0   [0.0s]
 2 |       3 |     3 / 3     |       4 |       4 |       4 |       4 |    1   direct union-find agrees: True; inversion-orbits split by untransposed profile: 0   [0.0s]
 3 |       9 |     6 / 6     |      10 |      10 |      10 |      10 |    3   direct union-find agrees: True; inversion-orbits split by untransposed profile: 0   [0.0s]
 4 |      37 |    16 / 16    |      35 |      35 |      35 |      35 |    6   direct union-find agrees: True; inversion-orbits split by untransposed profile: 0   [0.0s]
 5 |     177 |    34 / 34    |     102 |     101 |      97 |      97 |   13   direct union-find agrees: True; inversion-orbits split by untransposed profile: 0   [0.0s]
 6 |   1,054 |    95 / 92    |     395 |     

## 9. Section 8: how much the table determines (Table 3)
The measure is uniform on $X$. The entropies are exact finite sums over all labeled tables, weighted by the number of set partitions of each type.

In [10]:
def entropy(ps):
    return -sum(p * math.log2(p) for p in ps if p > 0)

for n in range(4, N_ENUM + 1):
    t0 = time.time(); N = factorial(n)
    B = sum(n_set_partitions_of_type(mu) for mu in partitions(n))
    H_lam_T = H_F_T = P_lam = P_F = 0.0; lam_marg = Counter(); F_marg = Counter()
    for mu in partitions(n):
        w = n_set_partitions_of_type(mu) / B
        for T in tables(mu):
            MV = M_counts(T); D = sum(MV.values()); pT = D / N
            mass_by_value = Counter()
            for lam, v in MV.items():
                mass_by_value[v] += v; lam_marg[lam] += w * v / N; F_marg[v] += w * v / N
            H_lam_T += w * pT * entropy([v / D for v in MV.values()])
            H_F_T += w * pT * entropy([c / D for c in mass_by_value.values()])
            P_lam += w * pT * (len(MV) == 1); P_F += w * pT * (len(mass_by_value) == 1)
    H_lam = entropy(lam_marg.values()); H_F = entropy(F_marg.values())
    print(f"n={n}: H(lambda)={H_lam:.3f}  H(lambda|T)={H_lam_T:.3f}  I(lambda;T)={H_lam - H_lam_T:.3f}  "
          f"H(F)={H_F:.3f}  H(F|T)={H_F_T:.3f}  P_lambda={P_lam:.4f}  P_F={P_F:.4f}   [{time.time() - t0:.1f}s]")

n=4: H(lambda)=2.094  H(lambda|T)=1.304  I(lambda;T)=0.790  H(F)=2.305  H(F|T)=0.554  P_lambda=0.0667  P_F=0.5000   [0.0s]
n=5: H(lambda)=2.557  H(lambda|T)=1.718  I(lambda;T)=0.839  H(F)=3.051  H(F|T)=0.828  P_lambda=0.0192  P_F=0.3750   [0.0s]
n=6: H(lambda)=3.035  H(lambda|T)=2.121  I(lambda;T)=0.914  H(F)=3.747  H(F|T)=1.136  P_lambda=0.0049  P_F=0.2332   [0.0s]
n=7: H(lambda)=3.436  H(lambda|T)=2.517  I(lambda;T)=0.919  H(F)=4.271  H(F|T)=1.465  P_lambda=0.0011  P_F=0.1349   [0.1s]


## 10. Section 8: level sets of $F$ (Table 4) and the values at $n=7$ (Appendix C)

In [11]:
for n in range(6, N_TABLE + 1):
    t0 = time.time(); level = defaultdict(list)
    for mu in partitions(n):
        for T in table_classes(mu, True)[1]:
            for lam, v in M_counts(T).items():
                level[v].append((mu, lam))
    profiles = sum(len(v) for v in level.values())
    across_types = sum(len({mu for mu, _ in cells}) > 1 for cells in level.values())
    across_cycles = sum(len({len(lam) for _, lam in cells}) > 1 for cells in level.values())
    top = max(level.items(), key=lambda kv: len(kv[1]))
    print(f"n={n}: profiles {profiles:,} | values {len(level)} | values shared by >=2 block types {across_types} | "
          f"by >=2 cycle counts {across_cycles} | largest level set F={top[0]} with {len(top[1]):,} profiles   [{time.time() - t0:.1f}s]")
    if n == 7:
        print("   values at n=7:", sorted(level))

n=6: profiles 344 | values 30 | values shared by >=2 block types 21 | by >=2 cycle counts 22 | largest level set F=2 with 53 profiles   [0.0s]
n=7: profiles 1,086 | values 60 | values shared by >=2 block types 40 | by >=2 cycle counts 43 | largest level set F=4 with 131 profiles   [0.3s]
   values at n=7: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15, 16, 18, 20, 21, 24, 25, 28, 30, 32, 36, 40, 45, 48, 50, 54, 60, 64, 70, 72, 84, 90, 96, 100, 105, 108, 120, 140, 144, 160, 168, 210, 216, 240, 260, 280, 288, 300, 324, 360, 420, 432, 480, 504, 540, 630, 720, 840]


## 11. Appendices A and B

In [12]:
print("Appendix A: table classes at n = 4 (without transpose)")
for mu in partitions(4):
    for T in sorted(table_classes(mu, False)[1], reverse=True):
        print(f"  mu={mu}  T={T}  |K sigma K|={fisher_yates(T)}  |M|:", dict(sorted(M_counts(T).items(), reverse=True)))
print()
print("Appendix B: n = 7 by block type: tables, classes ~, profiles, classes ~t, profiles (transpose-merged)")
tot = [0] * 5
for mu in partitions(7):
    nt, r1 = table_classes(mu, False); _, r2 = table_classes(mu, True)
    row = [nt, len(r1), sum(len(trail_spectra(t)) for t in r1), len(r2), sum(len(trail_spectra(t)) for t in r2)]
    tot = [a + b for a, b in zip(tot, row)]
    print(f"  {str(mu):<22}", row)
print("  totals:", tot)

Appendix A: table classes at n = 4 (without transpose)
  mu=(4,)  T=((4,),)  |K sigma K|=24  |M|: {(4,): 6, (3, 1): 8, (2, 2): 3, (2, 1, 1): 6, (1, 1, 1, 1): 1}
  mu=(3, 1)  T=((3, 0), (0, 1))  |K sigma K|=6  |M|: {(3, 1): 2, (2, 1, 1): 3, (1, 1, 1, 1): 1}
  mu=(3, 1)  T=((2, 1), (1, 0))  |K sigma K|=18  |M|: {(4,): 6, (3, 1): 6, (2, 2): 3, (2, 1, 1): 3}
  mu=(2, 2)  T=((2, 0), (0, 2))  |K sigma K|=4  |M|: {(2, 2): 1, (2, 1, 1): 2, (1, 1, 1, 1): 1}
  mu=(2, 2)  T=((1, 1), (1, 1))  |K sigma K|=16  |M|: {(4,): 4, (3, 1): 8, (2, 1, 1): 4}
  mu=(2, 2)  T=((0, 2), (2, 0))  |K sigma K|=4  |M|: {(4,): 2, (2, 2): 2}
  mu=(2, 1, 1)  T=((2, 0, 0), (0, 1, 0), (0, 0, 1))  |K sigma K|=2  |M|: {(2, 1, 1): 1, (1, 1, 1, 1): 1}
  mu=(2, 1, 1)  T=((2, 0, 0), (0, 0, 1), (0, 1, 0))  |K sigma K|=2  |M|: {(2, 2): 1, (2, 1, 1): 1}
  mu=(2, 1, 1)  T=((1, 1, 0), (1, 0, 0), (0, 0, 1))  |K sigma K|=4  |M|: {(3, 1): 2, (2, 1, 1): 2}
  mu=(2, 1, 1)  T=((1, 1, 0), (0, 0, 1), (1, 0, 0))  |K sigma K|=4  |M|: {(4,): 2